In [0]:
# Databricks Notebook: Transaction_Silver
# Cell 1: Fact Cleansing, Deduplication & Upsert

from pyspark.sql.functions import col, current_timestamp, to_timestamp, trim
from delta.tables import DeltaTable

# 1. Read Raw Transactions from Bronze
df_raw_txn = spark.table("bankingpoc.bronze.transaction")

# 2. Standardize Types, Clean Strings, and Deduplicate by Primary Key
df_clean_txn = df_raw_txn.select(
    col("transaction_id").cast("int"),
    col("account_id").cast("int"),
    trim(col("transaction_type")).alias("transaction_type"),
    col("amount").cast("decimal(18,2)"),
    to_timestamp(col("transaction_date")).alias("transaction_date"),
    trim(col("description")).alias("description"),
    current_timestamp().alias("silver_processed_timestamp")
).dropDuplicates(["transaction_id"])

target_txn_table = "bankingpoc.silver.transaction"

# 3. Write or Merge into Silver Delta Table
if not spark.catalog.tableExists(target_txn_table):
    df_clean_txn.write.format("delta").mode("overwrite").saveAsTable(target_txn_table)
    print(f"Initialized table: {target_txn_table}")
else:
    DeltaTable.forName(spark, target_txn_table).alias("tgt").merge(
        source=df_clean_txn.alias("src"),
        condition="tgt.transaction_id = src.transaction_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("Transaction upsert completed successfully.")

Initialized table: bankingpoc.silver.transaction
